# AiSH Qwen3.5-0.8B Local Notebook

This notebook tests `Qwen/Qwen3.5-0.8B` as the local AI-command-generation model for AiSH.

Target laptop:

```text
GPU: NVIDIA RTX 5060 Laptop GPU, 8GB VRAM
RAM: 24GB system RAM
Recommended OS path: Windows + PowerShell
```

AiSH model role:

```text
Normal Mode  -> no model
History Mode -> local history/project scorer
AI Mode      -> Qwen3.5-0.8B, manually triggered with Ctrl + Space
```

Do not run this model on every keystroke. Use it only when the user asks for an AI-generated command.


## 0. Local setup instructions

From the repo root on Windows PowerShell:

```powershell
cd qwen3.5-0.8b
py -3.11 -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install notebook ipykernel
python -m ipykernel install --user --name aish-qwen35-08b --display-name "AiSH Qwen3.5 0.8B"
jupyter notebook
```

Then open this notebook and select the kernel `AiSH Qwen3.5 0.8B`.

Linux/macOS alternative:

```bash
cd qwen3.5-0.8b
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install notebook ipykernel
python -m ipykernel install --user --name aish-qwen35-08b --display-name "AiSH Qwen3.5 0.8B"
jupyter notebook
```


## 1. Install dependencies

For the RTX 5060 Laptop GPU, start with recent CUDA-enabled PyTorch wheels. This notebook uses the CUDA 12.8 package index. If the install fails, use the official PyTorch selector and copy its latest Windows + Pip + CUDA command.


In [ ]:
%pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
%pip install --upgrade transformers accelerate safetensors huggingface_hub sentencepiece protobuf pillow ipywidgets
%pip install --upgrade datasets evaluate peft trl


## 2. Verify CUDA and GPU memory

Expected result: `CUDA available: True` and an RTX 5060 Laptop GPU with around 8GB VRAM. If CUDA is false, fix PyTorch/CUDA before loading the model.


In [ ]:
import platform
import torch

print('Python:', platform.python_version())
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(idx)
    print('GPU:', props.name)
    print(f'Total VRAM: {props.total_memory / (1024 ** 3):.2f} GB')
    print('CUDA runtime:', torch.version.cuda)
else:
    print('CPU fallback only. It will be much slower.')


## 3. Load Qwen3.5-0.8B

The first run downloads the model. Later runs use the Hugging Face cache. Start with short generations because AiSH only needs concise command suggestions.

If you hit VRAM errors: close GPU-heavy apps, reduce `MAX_NEW_TOKENS`, restart the kernel, and keep only one model loaded.


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = 'Qwen/Qwen3.5-0.8B'
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.25
TOP_P = 0.9

torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map='auto' if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.eval()

print('Loaded:', MODEL_ID)
print('Main device:', next(model.parameters()).device)
print('dtype:', next(model.parameters()).dtype)


## 4. AiSH command-generation prompt

The model should return compact JSON, not prose. AiSH can then show it as a command card and run a deterministic safety check before allowing the user to accept it.

Expected shape:

```json
{
  "command": "npm run dev",
  "confidence": 0.84,
  "shell": "powershell",
  "reason": "package.json has a dev script",
  "risk": "low"
}
```


In [ ]:
import json
import re
from typing import Dict, Any, List

SYSTEM_PROMPT = '''You are AiSH, a local terminal command assistant.
Convert the user's intent into ONE safe shell command for the given operating system and shell.

Rules:
- Return ONLY valid JSON.
- Do not wrap the JSON in markdown.
- Do not execute anything.
- Prefer simple, common commands.
- If the request is ambiguous, return an empty command and ask for clarification in the reason.
- If the command could modify, delete, publish, overwrite, or permanently change data, mark risk as high.
- Use the exact shell and OS context provided.
- Keep the command short.
'''

def build_aish_messages(user_intent: str, os_name: str = 'windows', shell: str = 'powershell', cwd: str = '.', project_type: str = 'unknown', detected_files: List[str] | None = None, recent_commands: List[str] | None = None) -> list:
    detected_files = detected_files or []
    recent_commands = recent_commands or []
    context = {
        'os': os_name,
        'shell': shell,
        'cwd': cwd,
        'project_type': project_type,
        'detected_files': detected_files,
        'recent_commands': recent_commands[-10:],
    }
    user_prompt = {
        'intent': user_intent,
        'context': context,
        'output_schema': {
            'command': 'string',
            'confidence': 'float between 0 and 1',
            'shell': shell,
            'reason': 'short explanation',
            'risk': 'low | medium | high',
        },
    }
    return [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user', 'content': [{'type': 'text', 'text': json.dumps(user_prompt, indent=2)}]},
    ]

def extract_json(text: str) -> Dict[str, Any]:
    text = text.strip()
    text = re.sub(r'^```(?:json)?', '', text).strip()
    text = re.sub(r'```$', '', text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        return {'command': '', 'confidence': 0.0, 'shell': '', 'reason': 'Model did not return JSON.', 'risk': 'high', 'raw': text}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {'command': '', 'confidence': 0.0, 'shell': '', 'reason': 'Model returned invalid JSON.', 'risk': 'high', 'raw': text}


## 5. Deterministic safety filter

The model's risk label is not enough. AiSH must check the generated command before displaying or accepting it. This is a first-pass local filter; the production app should be stricter.


In [ ]:
DANGEROUS_PATTERNS = [
    r'\brm\s+-rf\b',
    r'\bsudo\s+rm\b',
    r'\brmdir\s+/s\b',
    r'\bdel\s+/s\s+/q\b',
    r'\bgit\s+reset\s+--hard\b',
    r'\bdocker\s+system\s+prune\b',
    r'\bkubectl\s+delete\b',
    r'\bnpm\s+publish\b',
    r'\bchmod\s+-R\s+777\b',
    r'\bdrop\s+database\b',
    r'\bformat\b',
]

def safety_check(command: str) -> Dict[str, Any]:
    command_l = command.lower().strip()
    if not command_l:
        return {'allowed': False, 'risk': 'high', 'reason': 'Empty command.'}
    for pattern in DANGEROUS_PATTERNS:
        if re.search(pattern, command_l):
            return {'allowed': False, 'risk': 'high', 'reason': f'Matched dangerous pattern: {pattern}'}
    medium_markers = [' install ', ' add ', ' commit ', ' push ', ' move ', ' mv ', ' copy ', ' cp ', ' mkdir ', ' touch ', ' new-item ', ' set-item ', ' docker run ']
    padded = f' {command_l} '
    if any(marker in padded for marker in medium_markers):
        return {'allowed': True, 'risk': 'medium', 'reason': 'Command may modify local state. Show confirmation in AiSH.'}
    return {'allowed': True, 'risk': 'low', 'reason': 'No dangerous pattern detected.'}


## 6. Generate an AiSH command card

Production flow:

```text
User presses Ctrl + Space
AiSH builds shell/project context
Qwen3.5-0.8B generates JSON
AiSH safety filter checks command
AiSH shows command card
User accepts or rejects
```


In [ ]:
def generate_aish_command(user_intent: str, os_name: str = 'windows', shell: str = 'powershell', cwd: str = '.', project_type: str = 'unknown', detected_files: List[str] | None = None, recent_commands: List[str] | None = None, max_new_tokens: int = MAX_NEW_TOKENS) -> Dict[str, Any]:
    messages = build_aish_messages(user_intent, os_name, shell, cwd, project_type, detected_files, recent_commands)
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt')
    device = next(model.parameters()).device
    inputs = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs.items()}
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, pad_token_id=processor.tokenizer.eos_token_id if hasattr(processor, 'tokenizer') else None)
    input_len = inputs['input_ids'].shape[-1]
    text = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    parsed = extract_json(text)
    parsed['safety'] = safety_check(parsed.get('command', ''))
    return {'raw_text': text, 'parsed': parsed}

def to_command_card(result: Dict[str, Any]) -> Dict[str, Any]:
    parsed = result['parsed']
    safety = parsed.get('safety', {})
    return {
        'type': 'ai_command_card',
        'command': parsed.get('command', ''),
        'confidence': parsed.get('confidence', 0.0),
        'reason': parsed.get('reason', ''),
        'risk': safety.get('risk', parsed.get('risk', 'high')),
        'allowed': safety.get('allowed', False),
        'safety_reason': safety.get('reason', ''),
        'requires_confirmation': safety.get('risk') in {'medium', 'high'},
    }


## 7. Test examples

These examples are safe and close to AiSH's actual use cases.


In [ ]:
tests = [
    {
        'intent': 'show files in this folder sorted by newest first',
        'os_name': 'windows',
        'shell': 'powershell',
        'project_type': 'unknown',
        'detected_files': [],
        'recent_commands': ['Get-ChildItem', 'git status'],
    },
    {
        'intent': 'start this vite app',
        'os_name': 'windows',
        'shell': 'powershell',
        'project_type': 'node_vite_project',
        'detected_files': ['package.json', 'vite.config.ts'],
        'recent_commands': ['npm install', 'npm run dev'],
    },
    {
        'intent': 'show current git branch and changed files',
        'os_name': 'windows',
        'shell': 'powershell',
        'project_type': 'git_repo',
        'detected_files': ['.git'],
        'recent_commands': ['git status', 'git branch'],
    },
]

for t in tests:
    result = generate_aish_command(**t)
    card = to_command_card(result)
    print('\nIntent:', t['intent'])
    print(json.dumps(card, indent=2))


## 8. Local dataset format for future training

Use this format later for accepted/rejected AiSH suggestions. Keep private shell history local by default.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

sample_events = [
    {
        'id': 'evt_001',
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'os': 'windows',
        'shell': 'powershell',
        'cwd_hash': 'example_project_hash',
        'project_type': 'node_vite_project',
        'typed_prefix': 'npm',
        'intent': 'start this vite app',
        'command': 'npm run dev',
        'source': 'ai_generated',
        'accepted': True,
        'exit_code': 0,
    }
]

dataset_path = DATA_DIR / 'aish_command_events.jsonl'
with dataset_path.open('w', encoding='utf-8') as f:
    for event in sample_events:
        f.write(json.dumps(event) + '\n')

print('Wrote:', dataset_path)


## 9. Fine-tuning direction

Do not start with full fine-tuning. On 8GB VRAM, full fine-tuning is not the right path. Recommended order:

```text
1. Build deterministic completions.
2. Collect local accepted/rejected command events.
3. Train a tiny ranker for History Mode.
4. Only then test LoRA/QLoRA adapters for Qwen3.5-0.8B AI Mode.
```

The next cell only inspects possible trainable module names. Do not blindly fine-tune until the exact module names and memory behavior are known.


In [ ]:
module_names = []
for name, module in model.named_modules():
    if any(key in name.lower() for key in ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate', 'up_proj', 'down_proj', 'linear']):
        module_names.append(name)

print('Candidate module names:')
for name in module_names[:100]:
    print(name)
print(f'\nShown {min(len(module_names), 100)} of {len(module_names)} candidates.')


## 10. AiSH integration notes

Temporary development path:

```text
AiSH Desktop UI
  -> Rust backend command generate_ai_command()
  -> local Python model service using this notebook code
  -> Qwen3.5-0.8B
  -> JSON command card
```

Production direction:

```text
History Mode: deterministic completion engine + tiny ONNX ranker
AI Mode: Qwen3.5-0.8B or a future quantized local runtime
Trigger: Ctrl + Space, not every keystroke
Safety: deterministic filter before accept/execute
```
